# Cahn--Hilliard 2D: U-Net-only autoregressive baseline

This notebook trains and evaluates **only** a four-level 2D U-Net on
the exact Cahn--Hilliard artifact, split, and physical-error convention
used by the AWFNO benchmark.

**Protocol**

- Training examples are all adjacent pairs \(u^n\mapsto u^{n+1}\)
  from the selected training trajectories.
- At test time, the U-Net receives only the exact initial field
  \(u^0\). Every subsequent input is the preceding prediction.
- No exact intermediate state is used during rollout.
- The checkpoint is selected using the 30-trajectory validation set.
  The 70-trajectory test set is evaluated only after selection.
- The reported trajectory metric is the absolute physical
  \(L^2(\Omega\times[0,T])\), using the spatial rectangle rule and
  trapezoidal time quadrature used by the AWFNO benchmark.
- Training is full precision. AMP is intentionally disabled because
  the earlier mixed-precision U-Net run produced non-finite gradients.

This is a strong, problem-adapted U-Net baseline. Reflection padding
is used for the homogeneous Neumann boundary condition, GroupNorm is
used instead of batch-dependent normalization, and mass conservation
is imposed using only the current input field.

In [1]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


## Configuration

In [2]:
from pathlib import Path

CONFIG = {
    # Exact artifact used by the Cahn--Hilliard AWFNO experiment.
    "DATA_PATH": "/content/gdrive/MyDrive/anus/cahn_hilliard_train_grid_32x32.pt",
    # Use a new root so no checkpoint from the earlier AMP run is reused.
    "OUTPUT_ROOT": "/content/gdrive/MyDrive/anus/cahn_hilliard_unet_ar_only_stable_v1",

    # Exact common split.
    "TRAIN_POOL_SIZE": 400,
    "N_VAL": 30,
    "N_TEST": 70,
    "SPLIT_SEED": 2026,
    "SAMPLE_SIZES": [25, 50, 100, 200, 400],
    "RUN_SAMPLE_SIZES": [25, 50, 100, 200, 400],

    # Physical geometry and metric.
    "DT": 0.01,
    "USE_TRAPEZOIDAL_TIME_L2": True,
    "INCLUDE_T0_IN_FULL_L2": True,
    "ENFORCE_MASS": True,

    # Optimization. The learning rate is constant: no adaptive schedule.
    "EPOCHS": 200,
    "BATCH_SIZE": 32,
    "EVAL_BATCH_SIZE": 64,
    "LEARNING_RATE": 1.0e-3,
    "WEIGHT_DECAY": 1.0e-6,
    "GRAD_CLIP_NORM": 1.0,
    "VAL_ROLLOUT_INTERVAL": 10,
    "CHECKPOINT_INTERVAL": 10,
    "BASE_SEED": 1234,
    "NUM_WORKERS": 0,
    "PIN_MEMORY": True,
    "INFERENCE_REPEATS": 3,
    "FORCE_RETRAIN": False,
    "RUN_EXACT_HASH_OVERLAP_CHECK": True,

    # U-Net.
    "BASE_CHANNELS": 32,
    "LEVELS": 4,
    "NORM": "group",
    "PADDING_MODE": "reflect",

    # Outputs.
    "SAVE_ROLLOUTS": False,
    "SAVE_PLOTS": True,
    "SHOW_PLOTS": False,
}

DATA_PATH = Path(CONFIG["DATA_PATH"])
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"])
MODEL_ROOT = OUTPUT_ROOT / "models"
RESULT_ROOT = OUTPUT_ROOT / "results"
PLOT_ROOT = OUTPUT_ROOT / "plots"
for directory in (OUTPUT_ROOT, MODEL_ROOT, RESULT_ROOT, PLOT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

assert CONFIG["TRAIN_POOL_SIZE"] + CONFIG["N_VAL"] + CONFIG["N_TEST"] == 500
assert max(CONFIG["SAMPLE_SIZES"]) == CONFIG["TRAIN_POOL_SIZE"]

## Imports and deterministic setup

In [3]:
import copy
import gc
import hashlib
import json
import os
import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GiB):",
        torch.cuda.get_device_properties(0).total_memory / 2**30,
    )
    # TF32 is stable here and accelerates convolution on recent GPUs.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def torch_load_safe(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


with open(OUTPUT_ROOT / "config.json", "w") as stream:
    json.dump(json_ready(CONFIG), stream, indent=2)

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memory (GiB): 39.4935302734375


## Load the exact artifact and reproduce the fixed split

In [4]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Edit CONFIG['DATA_PATH'] if needed."
    )

raw_data = torch_load_safe(DATA_PATH, map_location="cpu")
if "X_input" not in raw_data or "Y_output" not in raw_data:
    raise KeyError("Expected keys 'X_input' and 'Y_output'.")

X_input = raw_data["X_input"].float()
Y_output = raw_data["Y_output"].float()
if X_input.ndim != 4 or X_input.shape[-1] != 1:
    raise ValueError(f"Expected X_input=(N,Nx,Ny,1), got {X_input.shape}.")
if Y_output.ndim != 5 or Y_output.shape[-1] != 1:
    raise ValueError(f"Expected Y_output=(N,T,Nx,Ny,1), got {Y_output.shape}.")

X_np = X_input[..., 0].contiguous().numpy().astype(np.float32)
Y_np = (
    Y_output[..., 0]
    .permute(0, 2, 3, 1)
    .contiguous()
    .numpy()
    .astype(np.float32)
)
U_ALL = np.concatenate([X_np[..., None], Y_np], axis=-1)
del X_input, Y_output, X_np, Y_np

if "x_grid" in raw_data:
    x_grid = raw_data["x_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "xg" in raw_data:
    x_grid = raw_data["xg"].float().reshape(-1).numpy().astype(np.float32)
else:
    x_grid = np.linspace(0.0, 1.0, U_ALL.shape[1], dtype=np.float32)

if "y_grid" in raw_data:
    y_grid = raw_data["y_grid"].float().reshape(-1).numpy().astype(np.float32)
elif "yg" in raw_data:
    y_grid = raw_data["yg"].float().reshape(-1).numpy().astype(np.float32)
else:
    y_grid = np.linspace(0.0, 1.0, U_ALL.shape[2], dtype=np.float32)

del raw_data
gc.collect()

N_TOTAL, NX, NY, NT = U_ALL.shape
if (NX, NY, NT) != (32, 32, 101):
    raise ValueError(f"Expected (N,32,32,101), received {U_ALL.shape}.")
if N_TOTAL < 500:
    raise ValueError(f"Need at least 500 trajectories, received {N_TOTAL}.")
if not np.all(np.isfinite(U_ALL)):
    raise ValueError("Dataset contains NaN or Inf.")

DX = float(np.mean(np.diff(x_grid.astype(np.float64))))
DY = float(np.mean(np.diff(y_grid.astype(np.float64))))
DT = float(CONFIG["DT"])
t_grid = np.arange(NT, dtype=np.float32) * DT

rng = np.random.default_rng(CONFIG["SPLIT_SEED"])
permutation = rng.permutation(N_TOTAL)
train_idx = permutation[: CONFIG["TRAIN_POOL_SIZE"]]
val_start = CONFIG["TRAIN_POOL_SIZE"]
val_stop = val_start + CONFIG["N_VAL"]
test_stop = val_stop + CONFIG["N_TEST"]
val_idx = permutation[val_start:val_stop]
test_idx = permutation[val_stop:test_stop]

U_TRAIN_POOL = np.ascontiguousarray(U_ALL[train_idx], dtype=np.float32)
U_VAL = np.ascontiguousarray(U_ALL[val_idx], dtype=np.float32)
U_TEST = np.ascontiguousarray(U_ALL[test_idx], dtype=np.float32)
del U_ALL
gc.collect()

split_path = OUTPUT_ROOT / "split_indices.npz"
if split_path.exists():
    saved_split = np.load(split_path)
    for key, current in (
        ("train_idx", train_idx),
        ("val_idx", val_idx),
        ("test_idx", test_idx),
    ):
        if not np.array_equal(saved_split[key], current):
            raise RuntimeError(
                f"Existing split disagrees for {key}; use a new OUTPUT_ROOT."
            )
else:
    np.savez(
        split_path,
        train_idx=train_idx,
        val_idx=val_idx,
        test_idx=test_idx,
        split_seed=np.asarray(CONFIG["SPLIT_SEED"]),
    )

print("\nLoaded Cahn--Hilliard benchmark")
print("--------------------------------")
print("all data       :", (N_TOTAL, NX, NY, NT))
print("training pool  :", U_TRAIN_POOL.shape)
print("validation     :", U_VAL.shape)
print("test           :", U_TEST.shape)
print("dx, dy, dt     :", DX, DY, DT)
print("train_idx first:", train_idx[:10])
print("val_idx first  :", val_idx[:10])
print("test_idx first :", test_idx[:10])


def trajectory_hashes(trajectories):
    output = set()
    for trajectory in trajectories:
        contiguous = np.ascontiguousarray(trajectory, dtype=np.float32)
        output.add(hashlib.sha256(contiguous.view(np.uint8)).hexdigest())
    return output


if CONFIG["RUN_EXACT_HASH_OVERLAP_CHECK"]:
    train_hash = trajectory_hashes(U_TRAIN_POOL)
    val_hash = trajectory_hashes(U_VAL)
    test_hash = trajectory_hashes(U_TEST)
    print("\nExact trajectory overlap")
    print("------------------------")
    print("train vs validation:", len(train_hash & val_hash))
    print("train vs test      :", len(train_hash & test_hash))
    print("validation vs test :", len(val_hash & test_hash))
    assert not (train_hash & val_hash)
    assert not (train_hash & test_hash)
    assert not (val_hash & test_hash)
    del train_hash, val_hash, test_hash


Loaded Cahn--Hilliard benchmark
--------------------------------
all data       : (520, 32, 32, 101)
training pool  : (400, 32, 32, 101)
validation     : (30, 32, 32, 101)
test           : (70, 32, 32, 101)
dx, dy, dt     : 0.06451612903225806 0.06451612903225806 0.01
train_idx first: [253 450 274  42 518 167 391 192  84 145]
val_idx first  : [285 136 466 366 375 317 156 186 358 148]
test_idx first : [110 237 469 121 234 226 109 164  43 102]

Exact trajectory overlap
------------------------
train vs validation: 0
train vs test      : 0
validation vs test : 0


## Physical error, mass conservation, and training data

In [5]:
def time_weights(nt):
    weights = np.ones(nt, dtype=np.float64)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = 0.5
        weights[-1] = 0.5
    return weights


def l2_per_timestep_np(prediction, truth):
    if prediction.shape != truth.shape:
        raise ValueError(f"Shape mismatch: {prediction.shape} vs {truth.shape}")
    n_samples = prediction.shape[0]
    output = np.empty((n_samples, prediction.shape[-1]), dtype=np.float64)
    for start in range(0, n_samples, 16):
        stop = min(start + 16, n_samples)
        error = (
            prediction[start:stop].astype(np.float64)
            - truth[start:stop].astype(np.float64)
        )
        output[start:stop] = np.sqrt(
            np.sum(error * error, axis=(1, 2)) * DX * DY + 1.0e-30
        )
    return output


def rollout_metrics(prediction, truth):
    l2_t = l2_per_timestep_np(prediction, truth)
    used = l2_t if CONFIG["INCLUDE_T0_IN_FULL_L2"] else l2_t[:, 1:]
    weights = time_weights(used.shape[1])[None, :]
    full = np.sqrt(np.sum(weights * used * used, axis=1) * DT + 1.0e-30)
    final = l2_t[:, -1]
    return {
        "full_mean": float(np.mean(full)),
        "full_std": float(np.std(full)),
        "full_median": float(np.median(full)),
        "final_mean": float(np.mean(final)),
        "final_std": float(np.std(final)),
        "final_median": float(np.median(final)),
        "mean_t": np.mean(l2_t, axis=0),
        "std_t": np.std(l2_t, axis=0),
        "median_t": np.median(l2_t, axis=0),
        "all_full": full,
        "all_final": final,
    }


def physical_field_l2_torch(prediction, truth):
    return torch.sqrt(
        torch.sum((prediction - truth) ** 2, dim=(-2, -1))
        * DX
        * DY
        + 1.0e-30
    )


def enforce_mass_torch(prediction, reference):
    if not CONFIG["ENFORCE_MASS"]:
        return prediction
    return (
        prediction
        + reference.mean(dim=(-2, -1), keepdim=True)
        - prediction.mean(dim=(-2, -1), keepdim=True)
    )


def mass_drift_np(prediction):
    mass = prediction.mean(axis=(1, 2))
    return np.max(np.abs(mass - mass[:, :1]), axis=1)


class AdjacentPairDataset2D(Dataset):
    def __init__(self, trajectories):
        self.trajectories = np.asarray(trajectories, dtype=np.float32)
        self.n_steps = self.trajectories.shape[-1] - 1

    def __len__(self):
        return self.trajectories.shape[0] * self.n_steps

    def __getitem__(self, index):
        trajectory = index // self.n_steps
        step = index % self.n_steps
        current = torch.from_numpy(
            self.trajectories[trajectory, :, :, step]
        )
        future = torch.from_numpy(
            self.trajectories[trajectory, :, :, step + 1]
        )
        return current, future


@dataclass
class ScalarNormalizer:
    mean: float
    std: float
    eps: float = 1.0e-6

    def encode(self, tensor):
        return (tensor - self.mean) / (self.std + self.eps)

    def decode(self, tensor):
        return tensor * (self.std + self.eps) + self.mean


def training_normalizer(trajectories):
    return ScalarNormalizer(
        mean=float(np.mean(trajectories, dtype=np.float64)),
        std=float(np.std(trajectories, dtype=np.float64)),
    )


def normalized_grid_channels(x_values, y_values):
    x = torch.tensor(x_values, dtype=torch.float32)
    y = torch.tensor(y_values, dtype=torch.float32)
    x = (x - x.min()) / (x.max() - x.min() + 1.0e-12)
    y = (y - y.min()) / (y.max() - y.min() + 1.0e-12)
    xx, yy = torch.meshgrid(x, y, indexing="ij")
    return torch.stack([xx, yy], dim=0)

## Four-level autoregressive U-Net

The input channels are the normalized current field and the two spatial
coordinates. The output is the normalized next field. The architecture
uses reflection padding and GroupNorm.

In [6]:
def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class UNetConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        def normalization(channels):
            if CONFIG["NORM"] == "group":
                return nn.GroupNorm(group_count(channels), channels)
            if CONFIG["NORM"] == "batch":
                return nn.BatchNorm2d(channels)
            if CONFIG["NORM"] == "none":
                return nn.Identity()
            raise ValueError(CONFIG["NORM"])

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=CONFIG["PADDING_MODE"],
            ),
            normalization(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=CONFIG["PADDING_MODE"],
            ),
            normalization(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UNet2D_AR(nn.Module):
    def __init__(self):
        super().__init__()
        base = int(CONFIG["BASE_CHANNELS"])
        if int(CONFIG["LEVELS"]) != 4:
            raise ValueError("This implementation uses exactly four levels.")
        channels = [base, 2 * base, 4 * base, 8 * base, 16 * base]
        self.register_buffer(
            "coordinates",
            normalized_grid_channels(x_grid, y_grid).unsqueeze(0),
            persistent=False,
        )
        self.encoders = nn.ModuleList(
            [
                UNetConvBlock(3, channels[0]),
                UNetConvBlock(channels[0], channels[1]),
                UNetConvBlock(channels[1], channels[2]),
                UNetConvBlock(channels[2], channels[3]),
            ]
        )
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = UNetConvBlock(channels[3], channels[4])
        self.upconvs = nn.ModuleList(
            [
                nn.ConvTranspose2d(
                    channels[4], channels[3], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[3], channels[2], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[2], channels[1], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[1], channels[0], kernel_size=2, stride=2
                ),
            ]
        )
        self.decoders = nn.ModuleList(
            [
                UNetConvBlock(2 * channels[3], channels[3]),
                UNetConvBlock(2 * channels[2], channels[2]),
                UNetConvBlock(2 * channels[1], channels[1]),
                UNetConvBlock(2 * channels[0], channels[0]),
            ]
        )
        self.output = nn.Conv2d(channels[0], 1, kernel_size=1)

    def forward(self, normalized_state):
        batch = normalized_state.shape[0]
        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        x = torch.cat([normalized_state.unsqueeze(1), coordinates], dim=1)
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for index, (upconv, decoder) in enumerate(
            zip(self.upconvs, self.decoders)
        ):
            x = upconv(x)
            x = torch.cat([x, skips[-(index + 1)]], dim=1)
            x = decoder(x)
        return self.output(x).squeeze(1)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


set_seed(CONFIG["BASE_SEED"])
audit_model = UNet2D_AR()
EXPECTED_PARAMETER_COUNT = count_parameters(audit_model)
print("U-Net parameters:", f"{EXPECTED_PARAMETER_COUNT:,}")
del audit_model
gc.collect()

U-Net parameters: 7,765,985


0

## Training, validation selection, and recursive rollout

In [7]:
def state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def predict_physical(model, current, normalizer):
    # Deliberately full precision: no autocast/AMP.
    normalized_prediction = model(normalizer.encode(current))
    prediction = normalizer.decode(normalized_prediction)
    return enforce_mass_torch(prediction, current)


def train_one_epoch(model, loader, optimizer, normalizer):
    model.train()
    loss_sum = 0.0
    l2_sum = 0.0
    count = 0
    for current, target in loader:
        current = current.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        prediction = predict_physical(model, current, normalizer)
        loss = F.mse_loss(
            normalizer.encode(prediction),
            normalizer.encode(target),
        )
        if not torch.isfinite(loss):
            raise FloatingPointError(
                "U-Net produced non-finite loss; checkpoint not saved."
            )
        loss.backward()
        gradient_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), float(CONFIG["GRAD_CLIP_NORM"])
        )
        if not torch.isfinite(gradient_norm):
            raise FloatingPointError(
                "U-Net produced non-finite gradients; checkpoint not saved."
            )
        optimizer.step()

        with torch.no_grad():
            field_l2 = physical_field_l2_torch(prediction, target)
        batch_size = current.shape[0]
        loss_sum += float(loss.detach().item()) * batch_size
        l2_sum += float(field_l2.sum().item())
        count += batch_size
    return loss_sum / count, l2_sum / count


@torch.no_grad()
def evaluate_pairs(model, loader, normalizer):
    model.eval()
    loss_sum = 0.0
    l2_sum = 0.0
    count = 0
    for current, target in loader:
        current = current.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        prediction = predict_physical(model, current, normalizer)
        loss = F.mse_loss(
            normalizer.encode(prediction),
            normalizer.encode(target),
        )
        if not torch.isfinite(loss) or not torch.isfinite(prediction).all():
            raise FloatingPointError("Non-finite validation values.")
        field_l2 = physical_field_l2_torch(prediction, target)
        batch_size = current.shape[0]
        loss_sum += float(loss.item()) * batch_size
        l2_sum += float(field_l2.sum().item())
        count += batch_size
    return loss_sum / count, l2_sum / count


@torch.no_grad()
def recursive_trajectory(model, trajectories, normalizer):
    model.eval()
    initial_states = trajectories[:, :, :, 0]
    n_samples = initial_states.shape[0]
    prediction = np.empty((n_samples, NX, NY, NT), dtype=np.float32)
    prediction[:, :, :, 0] = initial_states
    chunk_size = int(CONFIG["EVAL_BATCH_SIZE"])
    for start in range(0, n_samples, chunk_size):
        stop = min(start + chunk_size, n_samples)
        current = torch.as_tensor(
            initial_states[start:stop],
            dtype=torch.float32,
            device=DEVICE,
        )
        for step in range(1, NT):
            current = predict_physical(model, current, normalizer).float()
            if not torch.isfinite(current).all():
                raise FloatingPointError(
                    f"Non-finite recursive prediction at step {step}."
                )
            prediction[start:stop, :, :, step] = (
                current.detach().cpu().numpy()
            )
    return prediction


def validation_full_error(model, normalizer):
    prediction = recursive_trajectory(model, U_VAL, normalizer)
    value = rollout_metrics(prediction, U_VAL)["full_mean"]
    del prediction
    gc.collect()
    return value


def timed_test_prediction(model, normalizer):
    _ = recursive_trajectory(
        model, U_TEST[: min(2, len(U_TEST))], normalizer
    )
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    prediction = None
    timings = []
    for _ in range(int(CONFIG["INFERENCE_REPEATS"])):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = recursive_trajectory(model, U_TEST, normalizer)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    return prediction, float(np.median(timings)), timings

In [8]:
def fit_or_load(n_train):
    # The same initialization seed is used for all sample sizes.
    run_seed = int(CONFIG["BASE_SEED"]) + 20000
    run_directory = MODEL_ROOT / f"N{n_train}"
    run_directory.mkdir(parents=True, exist_ok=True)
    done_path = run_directory / "done.pt"
    best_path = run_directory / "best.pt"
    latest_path = run_directory / "latest.pt"

    if CONFIG["FORCE_RETRAIN"]:
        for path in (done_path, best_path, latest_path):
            if path.exists():
                path.unlink()

    normalizer = training_normalizer(U_TRAIN_POOL[:n_train])
    set_seed(run_seed)
    model = UNet2D_AR().to(DEVICE)
    if count_parameters(model) != EXPECTED_PARAMETER_COUNT:
        raise RuntimeError("Parameter count changed unexpectedly.")

    if done_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        print(f"Loading completed checkpoint: {done_path}")
        checkpoint = torch_load_safe(done_path, map_location=DEVICE)
        model.load_state_dict(checkpoint["model_state"])
        restored_normalizer = ScalarNormalizer(
            mean=float(checkpoint["normalizer"]["mean"]),
            std=float(checkpoint["normalizer"]["std"]),
        )
        return {
            "model": model,
            "normalizer": restored_normalizer,
            "history": checkpoint["history"],
            "best_epoch": int(checkpoint["best_epoch"]),
            "best_val_full": float(checkpoint["best_val_full"]),
            "train_time_sec": float(checkpoint["train_time_sec"]),
            "checkpoint": str(done_path),
        }

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(CONFIG["LEARNING_RATE"]),
        weight_decay=float(CONFIG["WEIGHT_DECAY"]),
    )

    train_dataset = AdjacentPairDataset2D(U_TRAIN_POOL[:n_train])
    validation_dataset = AdjacentPairDataset2D(U_VAL)
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(run_seed + 100000 + int(n_train))
    train_loader = DataLoader(
        train_dataset,
        batch_size=int(CONFIG["BATCH_SIZE"]),
        shuffle=True,
        generator=shuffle_generator,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"),
        drop_last=False,
    )
    validation_loader = DataLoader(
        validation_dataset,
        batch_size=int(CONFIG["EVAL_BATCH_SIZE"]),
        shuffle=False,
        num_workers=int(CONFIG["NUM_WORKERS"]),
        pin_memory=bool(CONFIG["PIN_MEMORY"] and DEVICE.type == "cuda"),
        drop_last=False,
    )

    history = {
        "epoch": [],
        "train_loss": [],
        "train_field_l2": [],
        "val_pair_loss": [],
        "val_field_l2": [],
        "val_full_l2": [],
    }
    best_epoch = -1
    best_val_full = float("inf")
    start_epoch = 1
    elapsed_before = 0.0

    if latest_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        latest = torch_load_safe(latest_path, map_location=DEVICE)
        if int(latest["n_train"]) != int(n_train):
            raise RuntimeError("Checkpoint sample-size mismatch.")
        model.load_state_dict(latest["model_state"])
        optimizer.load_state_dict(latest["optimizer_state"])
        optimizer_to_device(optimizer, DEVICE)
        history = latest["history"]
        best_epoch = int(latest["best_epoch"])
        best_val_full = float(latest["best_val_full"])
        start_epoch = int(latest["epoch"]) + 1
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        state = latest.get("shuffle_generator_state")
        if state is not None:
            if not torch.is_tensor(state):
                state = torch.as_tensor(state, dtype=torch.uint8)
            state = state.detach().cpu().to(dtype=torch.uint8)
            shuffle_generator.set_state(state)
        print(f"Resuming N={n_train} at epoch {start_epoch}.")

    print("\n" + "=" * 96)
    print(f"U-Net2D-AR-Neumann | N_train={n_train}")
    print("=" * 96)
    print("training trajectories :", U_TRAIN_POOL[:n_train].shape)
    print("training pairs        :", len(train_dataset))
    print("validation pairs      :", len(validation_dataset))
    print("parameters            :", f"{count_parameters(model):,}")
    print("batch size            :", CONFIG["BATCH_SIZE"])
    print("learning rate         :", CONFIG["LEARNING_RATE"], "(constant)")
    print("AMP                   : disabled")
    print("mass enforcement      :", CONFIG["ENFORCE_MASS"])
    print("test used for selection: no")

    segment_start = time.perf_counter()
    for epoch in range(start_epoch, int(CONFIG["EPOCHS"]) + 1):
        train_loss, train_l2 = train_one_epoch(
            model, train_loader, optimizer, normalizer
        )
        val_loss, val_l2 = evaluate_pairs(
            model, validation_loader, normalizer
        )
        run_rollout = (
            epoch % int(CONFIG["VAL_ROLLOUT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        )
        val_full = (
            validation_full_error(model, normalizer)
            if run_rollout
            else float("nan")
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_field_l2"].append(train_l2)
        history["val_pair_loss"].append(val_loss)
        history["val_field_l2"].append(val_l2)
        history["val_full_l2"].append(val_full)

        if run_rollout and np.isfinite(val_full) and val_full < best_val_full:
            best_epoch = int(epoch)
            best_val_full = float(val_full)
            torch.save(
                {
                    "n_train": int(n_train),
                    "model_state": state_dict_to_cpu(model),
                    "epoch": best_epoch,
                    "best_val_full": best_val_full,
                    "normalizer": {
                        "mean": normalizer.mean,
                        "std": normalizer.std,
                    },
                    "config": copy.deepcopy(CONFIG),
                },
                best_path,
            )

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{epoch:04d}/{CONFIG['EPOCHS']} | "
                f"train loss {train_loss:.6e} | "
                f"train L2 {train_l2:.6e} | "
                f"val pair loss {val_loss:.6e} | "
                f"val field L2 {val_l2:.6e} | "
                f"val full L2 {val_full:.6e}"
            )

        if (
            epoch % int(CONFIG["CHECKPOINT_INTERVAL"]) == 0
            or epoch == int(CONFIG["EPOCHS"])
        ):
            elapsed = elapsed_before + time.perf_counter() - segment_start
            torch.save(
                {
                    "n_train": int(n_train),
                    "epoch": int(epoch),
                    "model_state": state_dict_to_cpu(model),
                    "optimizer_state": optimizer.state_dict(),
                    "history": history,
                    "best_epoch": int(best_epoch),
                    "best_val_full": float(best_val_full),
                    "normalizer": {
                        "mean": normalizer.mean,
                        "std": normalizer.std,
                    },
                    "train_time_sec": float(elapsed),
                    "shuffle_generator_state": shuffle_generator.get_state(),
                    "config": copy.deepcopy(CONFIG),
                },
                latest_path,
            )

    total_train_time = elapsed_before + time.perf_counter() - segment_start
    if not best_path.exists():
        raise RuntimeError(f"No finite validation checkpoint for N={n_train}.")

    best = torch_load_safe(best_path, map_location=DEVICE)
    model.load_state_dict(best["model_state"])
    best_epoch = int(best["epoch"])
    best_val_full = float(best["best_val_full"])
    done_bundle = {
        "n_train": int(n_train),
        "model_state": state_dict_to_cpu(model),
        "history": history,
        "best_epoch": best_epoch,
        "best_val_full": best_val_full,
        "train_time_sec": float(total_train_time),
        "normalizer": {
            "mean": normalizer.mean,
            "std": normalizer.std,
        },
        "parameters": int(count_parameters(model)),
        "config": copy.deepcopy(CONFIG),
    }
    torch.save(done_bundle, done_path)
    print(
        f"Selected epoch {best_epoch} with validation full L2 "
        f"{best_val_full:.6e}"
    )
    return {
        "model": model,
        "normalizer": normalizer,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_full": best_val_full,
        "train_time_sec": float(total_train_time),
        "checkpoint": str(done_path),
    }

## Plots and sample-size sweep

In [9]:
def plot_training_history(history, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    epochs = np.asarray(history["epoch"], dtype=int)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    axes[0].semilogy(
        epochs, history["train_field_l2"], label="train field L2"
    )
    axes[0].semilogy(
        epochs, history["val_field_l2"], label="validation field L2"
    )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("absolute physical field L2")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    val_full = np.asarray(history["val_full_l2"], dtype=float)
    finite = np.isfinite(val_full)
    axes[1].semilogy(
        epochs[finite],
        val_full[finite],
        marker="o",
        label="validation trajectory L2",
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("absolute physical trajectory L2")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    figure.suptitle(f"U-Net2D-AR-Neumann, N={n_train}")
    figure.tight_layout()
    figure.savefig(
        PLOT_ROOT / f"N{n_train}_unet_training.png", dpi=170
    )
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)


def plot_l2_per_time(metrics, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], linewidth=2, label="mean")
    axis.plot(t_grid, metrics["median_t"], "--", linewidth=2, label="median")
    axis.fill_between(
        t_grid,
        np.maximum(metrics["mean_t"] - metrics["std_t"], 0.0),
        metrics["mean_t"] + metrics["std_t"],
        alpha=0.2,
        label="mean ± std",
    )
    axis.set_xlabel("time")
    axis.set_ylabel("absolute physical L2")
    axis.set_title(f"U-Net2D-AR-Neumann, N={n_train}")
    axis.grid(True, alpha=0.3)
    axis.legend()
    figure.tight_layout()
    figure.savefig(
        PLOT_ROOT / f"N{n_train}_unet_l2_per_time.png", dpi=170
    )
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)


def plot_solution_panel(prediction, n_train, sample=0):
    if not CONFIG["SAVE_PLOTS"]:
        return
    time_indices = [0, 10, 25, 50, 100]
    figure, axes = plt.subplots(
        3,
        len(time_indices),
        figsize=(3.0 * len(time_indices), 8.0),
        constrained_layout=True,
    )
    truth = U_TEST[sample]
    field_limit = max(
        float(np.max(np.abs(truth))),
        float(np.max(np.abs(prediction[sample]))),
    )
    error_limit = max(
        float(
            np.max(
                np.abs(
                    prediction[sample, :, :, index]
                    - truth[:, :, index]
                )
            )
        )
        for index in time_indices
    )
    for column, time_index in enumerate(time_indices):
        exact = truth[:, :, time_index]
        predicted = prediction[sample, :, :, time_index]
        error = predicted - exact
        axes[0, column].imshow(
            exact.T,
            origin="lower",
            cmap="coolwarm",
            vmin=-field_limit,
            vmax=field_limit,
        )
        axes[1, column].imshow(
            predicted.T,
            origin="lower",
            cmap="coolwarm",
            vmin=-field_limit,
            vmax=field_limit,
        )
        axes[2, column].imshow(
            error.T,
            origin="lower",
            cmap="RdBu_r",
            vmin=-error_limit,
            vmax=error_limit,
        )
        axes[0, column].set_title(f"t={t_grid[time_index]:.2f}")
        for row in range(3):
            axes[row, column].set_xticks([])
            axes[row, column].set_yticks([])
    axes[0, 0].set_ylabel("reference")
    axes[1, 0].set_ylabel("prediction")
    axes[2, 0].set_ylabel("error")
    figure.suptitle(f"U-Net2D-AR-Neumann, N={n_train}")
    figure.savefig(
        PLOT_ROOT / f"N{n_train}_unet_solution_panel.png", dpi=170
    )
    if CONFIG["SHOW_PLOTS"]:
        plt.show()
    plt.close(figure)

In [10]:
all_results = []
for n_train in CONFIG["RUN_SAMPLE_SIZES"]:
    if n_train not in CONFIG["SAMPLE_SIZES"]:
        raise ValueError(f"Unexpected sample size {n_train}.")

    fitted = fit_or_load(n_train)
    prediction, inference_median, inference_all = timed_test_prediction(
        fitted["model"], fitted["normalizer"]
    )
    metrics = rollout_metrics(prediction, U_TEST)
    mass_drift = mass_drift_np(prediction)
    result = {
        "n_train": int(n_train),
        "model": "U-Net2D-AR-Neumann",
        "protocol": "one_step_autoregressive",
        "parameters": int(count_parameters(fitted["model"])),
        "best_epoch": int(fitted["best_epoch"]),
        "validation_full_l2": float(fitted["best_val_full"]),
        "full_mean": metrics["full_mean"],
        "full_std": metrics["full_std"],
        "full_median": metrics["full_median"],
        "final_mean": metrics["final_mean"],
        "final_std": metrics["final_std"],
        "final_median": metrics["final_median"],
        "mass_drift_mean": float(np.mean(mass_drift)),
        "mass_drift_max": float(np.max(mass_drift)),
        "train_time_sec": float(fitted["train_time_sec"]),
        "inference_time_median_sec": inference_median,
        "inference_time_all_sec": inference_all,
        "checkpoint": fitted["checkpoint"],
    }
    all_results.append(result)

    np.savez_compressed(
        RESULT_ROOT / f"N{n_train}_unet_metrics.npz",
        mean_t=metrics["mean_t"],
        std_t=metrics["std_t"],
        median_t=metrics["median_t"],
        all_full=metrics["all_full"],
        all_final=metrics["all_final"],
        mass_drift=mass_drift,
        inference_times=np.asarray(inference_all),
    )
    if CONFIG["SAVE_ROLLOUTS"]:
        np.save(
            RESULT_ROOT / f"N{n_train}_unet_rollout.npy",
            prediction.astype(np.float32),
        )

    plot_training_history(fitted["history"], n_train)
    plot_l2_per_time(metrics, n_train)
    plot_solution_panel(prediction, n_train, sample=0)

    print("\nTest result")
    print("-----------")
    print("model             : U-Net2D-AR-Neumann")
    print("N_train           :", n_train)
    print("parameters        :", f"{result['parameters']:,}")
    print(
        "full mean/std     :",
        f"{result['full_mean']:.6e}",
        "/",
        f"{result['full_std']:.6e}",
    )
    print("full median       :", f"{result['full_median']:.6e}")
    print(
        "final mean/std    :",
        f"{result['final_mean']:.6e}",
        "/",
        f"{result['final_std']:.6e}",
    )
    print("final median      :", f"{result['final_median']:.6e}")
    print("mean mass drift   :", f"{result['mass_drift_mean']:.6e}")
    print("training time     :", f"{result['train_time_sec']:.2f} s")
    print(
        "inference median  :",
        f"{result['inference_time_median_sec']:.4f} s",
    )

    del fitted, prediction, metrics, mass_drift
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

Loading completed checkpoint: /content/gdrive/MyDrive/anus/cahn_hilliard_unet_ar_only_stable_v1/models/N25/done.pt

Test result
-----------
model             : U-Net2D-AR-Neumann
N_train           : 25
parameters        : 7,765,985
full mean/std     : 7.686477e-01 / 1.676284e-01
full median       : 7.680734e-01
final mean/std    : 1.160192e+00 / 3.095534e-01
final median      : 1.158461e+00
mean mass drift   : 9.897144e-08
training time     : 164.35 s
inference median  : 0.9811 s
Loading completed checkpoint: /content/gdrive/MyDrive/anus/cahn_hilliard_unet_ar_only_stable_v1/models/N50/done.pt

Test result
-----------
model             : U-Net2D-AR-Neumann
N_train           : 50
parameters        : 7,765,985
full mean/std     : 7.677136e-01 / 1.722578e-01
full median       : 7.816071e-01
final mean/std    : 1.165930e+00 / 3.081426e-01
final median      : 1.197931e+00
mean mass drift   : 9.711740e-08
training time     : 201.87 s
inference median  : 0.9935 s
Loading completed checkpoint: 

## Paper-ready summary

In [11]:
summary = pd.DataFrame(all_results).sort_values("n_train")
csv_path = RESULT_ROOT / "cahn_hilliard_unet_ar_summary.csv"
json_path = RESULT_ROOT / "cahn_hilliard_unet_ar_summary.json"
summary.to_csv(csv_path, index=False)
with open(json_path, "w") as stream:
    json.dump(json_ready(summary.to_dict(orient="records")), stream, indent=2)

display(
    summary[
        [
            "n_train",
            "model",
            "parameters",
            "full_mean",
            "full_std",
            "full_median",
            "final_mean",
            "final_std",
            "final_median",
            "train_time_sec",
            "inference_time_median_sec",
        ]
    ]
)

print("\n" + "=" * 132)
print("CAHN--HILLIARD 2D U-NET AUTOREGRESSIVE: ABSOLUTE PHYSICAL L2")
print("=" * 132)
for row in all_results:
    print(
        f"N={row['n_train']:4d} | "
        f"full mean {row['full_mean']:.6e} | "
        f"std {row['full_std']:.6e} | "
        f"median {row['full_median']:.6e} | "
        f"final mean {row['final_mean']:.6e} | "
        f"final median {row['final_median']:.6e} | "
        f"params {row['parameters']:,} | "
        f"train {row['train_time_sec']:.2f}s | "
        f"inference {row['inference_time_median_sec']:.4f}s"
    )
print("=" * 132)
print("Saved CSV :", csv_path)
print("Saved JSON:", json_path)

,n_train,model,parameters,full_mean,full_std,full_median,final_mean,final_std,final_median,train_time_sec,inference_time_median_sec
0,25,U-Net2D-AR-Neumann,7765985,0.768648,0.167628,0.768073,1.160192,0.309553,1.158461,164.351650,0.981055
1,50,U-Net2D-AR-Neumann,7765985,0.767714,0.172258,0.781607,1.165930,0.308143,1.197931,201.873236,0.993498
2,100,U-Net2D-AR-Neumann,7765985,0.677790,0.172525,0.669352,1.038364,0.311207,1.053852,375.018740,1.007554
3,200,U-Net2D-AR-Neumann,7765985,0.664534,0.172162,0.634815,1.027739,0.314760,1.011002,717.747915,0.990112
4,400,U-Net2D-AR-Neumann,7765985,0.632456,0.171092,0.632915,0.987006,0.307958,1.004788,1413.549595,0.988191



CAHN--HILLIARD 2D U-NET AUTOREGRESSIVE: ABSOLUTE PHYSICAL L2
N=  25 | full mean 7.686477e-01 | std 1.676284e-01 | median 7.680734e-01 | final mean 1.160192e+00 | final median 1.158461e+00 | params 7,765,985 | train 164.35s | inference 0.9811s
N=  50 | full mean 7.677136e-01 | std 1.722578e-01 | median 7.816071e-01 | final mean 1.165930e+00 | final median 1.197931e+00 | params 7,765,985 | train 201.87s | inference 0.9935s
N= 100 | full mean 6.777899e-01 | std 1.725246e-01 | median 6.693517e-01 | final mean 1.038364e+00 | final median 1.053852e+00 | params 7,765,985 | train 375.02s | inference 1.0076s
N= 200 | full mean 6.645342e-01 | std 1.721624e-01 | median 6.348150e-01 | final mean 1.027739e+00 | final median 1.011002e+00 | params 7,765,985 | train 717.75s | inference 0.9901s
N= 400 | full mean 6.324559e-01 | std 1.710921e-01 | median 6.329154e-01 | final mean 9.870056e-01 | final median 1.004788e+00 | params 7,765,985 | train 1413.55s | inference 0.9882s
Saved CSV : /content/gdrive

## Optional: release the Colab runtime

In [12]:
# Uncomment only after checking that the summary and checkpoints exist
# in Google Drive.
#
# from google.colab import runtime
# runtime.unassign()

In [13]:
# Uncomment only after checking that the summary and checkpoints exist
# in Google Drive.
#
from google.colab import runtime
runtime.unassign()